# Perceive: emotion + problem-type classifier

**Task.** Given the dialogue history up to a seeker turn, predict the dialogue's `emotion` and `problem` (two heads on one DistilRoBERTa encoder).

**Input.** `text` from `data/processed/{train,val,test}.jsonl` (made by `src/prep.py`): `speaker: text | speaker: text | ...`, ending on a seeker turn. Truncated **from the left** to 512 tokens, so the latest seeker message always survives.

**Loss.** Cross-entropy per head, summed, optionally class-weighted (weights from the train split only).

**Model selection.** Best epoch by mean of the two validation macro-F1 scores. The test split is only touched once, at the end.

## Platforms
This branch (`main`) runs locally. For GPU training use the **`kaggle`** or **`colab`** branch. They differ from `main` only in the path-setup block of the first code cell and in this note.

In [ ]:
import json, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report

CFG = dict(
    model_name="distilroberta-base",
    max_len=512,            # hard limit of RoBERTa-style encoders
    batch_size=16,
    lr=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    epochs=4,
    dropout=0.1,
    seed=42,
    class_weights=True,     # set False to compare against unweighted training
    weight_power=0.5,       # 1.0 = plain inverse frequency; 0.5 = tempered (sqrt), less extreme for tiny classes
    smoke_test=False,       # True: tiny subset + 1 epoch, just to check the pipeline runs
)
DATA_DIR = Path("data/processed")
OUT_DIR = DATA_DIR                               # prediction files are written next to the data
CKPT_DIR = Path("checkpoints/perceive")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("data:", DATA_DIR, "| outputs:", OUT_DIR, "| checkpoints:", CKPT_DIR)
import hashlib   # same value on every platform = same splits
print("splits.json sha256:", hashlib.sha256((DATA_DIR / "splits.json").read_bytes()).hexdigest()[:12])

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(CFG["seed"])

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
import transformers, sklearn
print("torch", torch.__version__, "| transformers", transformers.__version__, "| sklearn", sklearn.__version__)

## 1. Load data and label maps
Each row carries the dialogue's labels as text; `label_maps.json` converts them to integer ids.

In [ ]:
def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

data = {s: read_jsonl(DATA_DIR / f"{s}.jsonl") for s in ("train", "val", "test")}
maps = json.load(open(DATA_DIR / "label_maps.json"))
emo2id, prob2id = maps["emotion2id"], maps["problem2id"]
id2emo = {v: k for k, v in emo2id.items()}
id2prob = {v: k for k, v in prob2id.items()}

if CFG["smoke_test"]:
    rng = random.Random(0)
    for s, n in (("train", 300), ("val", 100), ("test", 100)):
        data[s] = rng.sample(data[s], n)
    CFG["epochs"] = 1

for rows in data.values():
    for r in rows:
        r["y_emo"] = emo2id[r["emotion"]]
        r["y_prob"] = prob2id[r["problem"]]

print({s: len(rows) for s, rows in data.items()})
print("emotion classes:", emo2id)
print("problem classes:", prob2id)

## 2. Tokenizer: left-truncation to 512 tokens
`truncation_side="left"` drops the **oldest** turns when the history is too long. The default (right) would drop the latest seeker message, which is the most informative part.

In [ ]:
tok = AutoTokenizer.from_pretrained(CFG["model_name"], truncation_side="left")

# how many training rows get truncated?
lengths = [len(ids) for ids in tok([r["text"] for r in data["train"]])["input_ids"]]
over = sum(l > CFG["max_len"] for l in lengths)
print(f"train rows over {CFG['max_len']} tokens: {over} ({over/len(lengths):.1%}); longest: {max(lengths)} tokens")

# sanity check: the END of the longest history must survive truncation
longest = max(data["train"], key=lambda r: len(r["text"]))
enc = tok(longest["text"], truncation=True, max_length=CFG["max_len"])
decoded = tok.decode(enc["input_ids"], skip_special_tokens=True, clean_up_tokenization_spaces=False)
assert len(enc["input_ids"]) == CFG["max_len"]
assert longest["text"][-40:] in decoded, "latest turn was cut, truncation side is wrong"
print("truncation check passed: latest seeker turn is kept")

## 3. Dataset / batching
Tokenisation happens per batch (dynamic padding), so short histories are not padded to 512.

In [ ]:
def collate(rows):
    enc = tok([r["text"] for r in rows], truncation=True, max_length=CFG["max_len"],
              padding=True, return_tensors="pt")
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "y_emo": torch.tensor([r["y_emo"] for r in rows]),
        "y_prob": torch.tensor([r["y_prob"] for r in rows]),
    }

g = torch.Generator().manual_seed(CFG["seed"])
train_loader = DataLoader(data["train"], batch_size=CFG["batch_size"], shuffle=True,
                          collate_fn=collate, generator=g)

## 4. Class weights (train split only)
`weight_c = (N / (K * n_c)) ** power`. With `power=1` a class with ~40 rows would get a weight near 38, which can destabilise training, so the default is the tempered square-root version. Set `class_weights=False` to compare.

In [ ]:
def class_weights(ys, n_classes, power):
    counts = np.bincount(ys, minlength=n_classes).clip(min=1)
    w = (len(ys) / (n_classes * counts)) ** power
    return torch.tensor(w, dtype=torch.float)

n_emo, n_prob = len(emo2id), len(prob2id)
if CFG["class_weights"]:
    w_emo = class_weights([r["y_emo"] for r in data["train"]], n_emo, CFG["weight_power"])
    w_prob = class_weights([r["y_prob"] for r in data["train"]], n_prob, CFG["weight_power"])
else:
    w_emo, w_prob = torch.ones(n_emo), torch.ones(n_prob)
print("emotion weights:", {id2emo[i]: round(float(w), 2) for i, w in enumerate(w_emo)})
print("problem weights:", {id2prob[i]: round(float(w), 2) for i, w in enumerate(w_prob)})

loss_emo = nn.CrossEntropyLoss(weight=w_emo.to(device))
loss_prob = nn.CrossEntropyLoss(weight=w_prob.to(device))

## 5. Model
One shared DistilRoBERTa encoder; the `<s>` token's final hidden state feeds two linear heads (emotion, problem).

In [ ]:
class PerceiveModel(nn.Module):
    def __init__(self, name, n_emo, n_prob, dropout):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        hidden = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.emo_head = nn.Linear(hidden, n_emo)
        self.prob_head = nn.Linear(hidden, n_prob)

    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        h = self.drop(h)
        return self.emo_head(h), self.prob_head(h)

model = PerceiveModel(CFG["model_name"], n_emo, n_prob, CFG["dropout"]).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## 6. Evaluation helpers
Macro-F1 is computed over the classes that actually occur in the evaluation set (a class with zero true examples, e.g. emotion `other` in val/test, cannot be scored).

In [ ]:
@torch.no_grad()
def predict(model, rows, batch_size=64):
    """Returns softmax probabilities for both heads, shape (n, n_emo) and (n, n_prob)."""
    model.eval()
    loader = DataLoader(rows, batch_size=batch_size, shuffle=False, collate_fn=collate)
    emo_p, prob_p = [], []
    for b in loader:
        e, p = model(b["input_ids"].to(device), b["attention_mask"].to(device))
        emo_p.append(e.float().softmax(-1).cpu()); prob_p.append(p.float().softmax(-1).cpu())
    return torch.cat(emo_p).numpy(), torch.cat(prob_p).numpy()

def scores(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    present = sorted(set(y_true.tolist()))
    return {
        "n": len(y_true),
        "acc": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=present, average="weighted", zero_division=0),
    }

def evaluate(rows, emo_p, prob_p):
    ye = [r["y_emo"] for r in rows]; yp = [r["y_prob"] for r in rows]
    return {"emotion": scores(ye, emo_p.argmax(1)), "problem": scores(yp, prob_p.argmax(1))}

## 7. Train
AdamW + linear warmup/decay, gradient clipping, fp16 autocast on CUDA. After every epoch we score the validation split and keep the best checkpoint (mean of emotion and problem macro-F1).

In [ ]:
steps = len(train_loader) * CFG["epochs"]
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup_ratio"] * steps), steps)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler(enabled=use_amp)

history, best_score = [], -1.0
for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    t0, running = time.time(), 0.0
    for b in tqdm(train_loader, desc=f"epoch {epoch}/{CFG['epochs']}"):
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            e_logits, p_logits = model(b["input_ids"], b["attention_mask"])
        loss = loss_emo(e_logits.float(), b["y_emo"]) + loss_prob(p_logits.float(), b["y_prob"])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        running += loss.item()

    val_e, val_p = predict(model, data["val"])
    m = evaluate(data["val"], val_e, val_p)
    score = (m["emotion"]["macro_f1"] + m["problem"]["macro_f1"]) / 2
    history.append({"epoch": epoch, "train_loss": running / len(train_loader), "val": m, "val_score": score})
    print(f"epoch {epoch}: loss {running/len(train_loader):.4f} | "
          f"val emo acc {m['emotion']['acc']:.3f} F1 {m['emotion']['macro_f1']:.3f} | "
          f"val prob acc {m['problem']['acc']:.3f} F1 {m['problem']['macro_f1']:.3f} | "
          f"{time.time()-t0:.0f}s")
    if score > best_score:
        best_score = score
        torch.save(model.state_dict(), CKPT_DIR / "best.pt")
        print(f"  saved best checkpoint (val score {score:.3f})")

## 8. Final evaluation with the best checkpoint
Overall, in-scope subset, and accuracy by seeker-turn position. The test split is scored once, here.

In [ ]:
model.load_state_dict(torch.load(CKPT_DIR / "best.pt", map_location=device))
preds = {s: predict(model, data[s]) for s in ("val", "test")}

BINS = [(1, 1), (2, 2), (3, 3), (4, 5), (6, 8), (9, 999)]

def full_report(split):
    rows = data[split]; emo_p, prob_p = preds[split]
    out = {"overall": evaluate(rows, emo_p, prob_p)}
    idx = [i for i, r in enumerate(rows) if r["in_scope"]]
    out["in_scope"] = evaluate([rows[i] for i in idx], emo_p[idx], prob_p[idx])
    out["by_turn"] = {}
    for lo, hi in BINS:
        idx = [i for i, r in enumerate(rows) if lo <= r["turn_idx"] <= hi]
        if idx:
            label = f"{lo}" if lo == hi else (f"{lo}+" if hi == 999 else f"{lo}-{hi}")
            out["by_turn"][label] = evaluate([rows[i] for i in idx], emo_p[idx], prob_p[idx])
    return out

reports = {s: full_report(s) for s in ("val", "test")}

for s in ("val", "test"):
    r = reports[s]
    print(f"\n===== {s} =====")
    for part in ("overall", "in_scope"):
        e, p = r[part]["emotion"], r[part]["problem"]
        print(f"{part:9s} n={e['n']:5d} | emotion acc {e['acc']:.3f} macroF1 {e['macro_f1']:.3f} | "
              f"problem acc {p['acc']:.3f} macroF1 {p['macro_f1']:.3f}")
    print("accuracy by seeker-turn position (emotion / problem):")
    for k, v in r["by_turn"].items():
        print(f"  turn {k:>4s}  n={v['emotion']['n']:5d}  {v['emotion']['acc']:.3f} / {v['problem']['acc']:.3f}")

In [ ]:
# per-class detail on the test split
rows = data["test"]; emo_p, prob_p = preds["test"]
print("EMOTION"); print(classification_report([r["y_emo"] for r in rows], emo_p.argmax(1),
      labels=sorted(set(r["y_emo"] for r in rows)),
      target_names=[id2emo[i] for i in sorted(set(r["y_emo"] for r in rows))], zero_division=0))
print("PROBLEM"); print(classification_report([r["y_prob"] for r in rows], prob_p.argmax(1),
      labels=sorted(set(r["y_prob"] for r in rows)),
      target_names=[id2prob[i] for i in sorted(set(r["y_prob"] for r in rows))], zero_division=0))

## 9. Does smoothing over turns help? (checked on val, applied to test)
The label is constant within a dialogue, so we can average the class probabilities over all seeker turns so far instead of trusting the latest turn alone. This only changes inference, not training.

In [ ]:
def smooth(rows, probs):
    """Running mean of probabilities over earlier turns of the same dialogue."""
    out = probs.copy()
    by_dlg = {}
    for i, r in enumerate(rows):
        by_dlg.setdefault(r["dialogue_id"], []).append(i)
    for idxs in by_dlg.values():
        idxs.sort(key=lambda i: rows[i]["turn_idx"])
        out[idxs] = np.cumsum(probs[idxs], axis=0) / np.arange(1, len(idxs) + 1)[:, None]
    return out

for s in ("val", "test"):
    rows = data[s]; emo_p, prob_p = preds[s]
    raw = evaluate(rows, emo_p, prob_p)
    sm = evaluate(rows, smooth(rows, emo_p), smooth(rows, prob_p))
    print(f"{s}: emotion acc {raw['emotion']['acc']:.3f} -> {sm['emotion']['acc']:.3f} | "
          f"problem acc {raw['problem']['acc']:.3f} -> {sm['problem']['acc']:.3f}  (raw -> smoothed)")

## 10. Save outputs
* `checkpoints/perceive/run.json`: hyper-parameters, per-epoch history, final reports (for the hyper-parameter table in the report).
* `perceive_preds_{val,test}.jsonl` (in `data/processed/` locally, `/kaggle/working/` on Kaggle): predicted probabilities per row, so later stages (Speak, Plan) can be tested on **predicted** emotion, as the proposal requires.

In [ ]:
(CKPT_DIR / "run.json").write_text(json.dumps(
    {"config": CFG, "history": history, "reports": reports}, indent=1, default=float))

for s in ("val", "test"):
    emo_p, prob_p = preds[s]
    with open(OUT_DIR / f"perceive_preds_{s}.jsonl", "w") as f:
        for r, e, p in zip(data[s], emo_p, prob_p):
            f.write(json.dumps({
                "dialogue_id": r["dialogue_id"], "turn_idx": r["turn_idx"],
                "emotion_pred": id2emo[int(e.argmax())], "problem_pred": id2prob[int(p.argmax())],
                "emotion_probs": [round(float(x), 4) for x in e],
                "problem_probs": [round(float(x), 4) for x in p],
            }) + "\n")
print("saved run.json and prediction files")